# Translate PPMI LEDD Concomitant_Medication_Log into structured data
- Clean, parse, and standardize free-text medication records

In [1]:
import pandas as pd
import re 
import matplotlib.pyplot as plt
from rapidfuzz import process, fuzz
import numpy as np
import unicodedata 

# Load raw LEDD concomitant medication data

In [2]:
dir = '/Users/emudr/Desktop/data/PPMI_LEDD_Project/'
outdir = '/Users/emudr/Desktop/'
file = 'LEDD_Concomitant_Medication_Log.csv'
df = pd.read_csv(dir+file)
df = df.drop(columns=['PAG_NAME', 'SEQNO', 'LEDDOSSTR', 'LAST_UPDATE']) # Remove unneccessary columns

In [3]:
def normalize_unicode(text):
    if pd.isna(text):
        return text
    text = text.replace("–", "-").replace("—", "-")
    return unicodedata.normalize("NFKC", str(text))

df["note_clean"] = df["LEDTRT"].apply(normalize_unicode)


df.to_csv('/Users/emudr/Desktop/unicode.csv')
diff_mask = df["LEDTRT"] != df["note_clean"]

df_diff = df[diff_mask]
print(df_diff)
#df.loc[diff_mask, ["col1", "col2"]]

jaentl

Empty DataFrame
Columns: [REC_ID, PATNO, EVENT_ID, INFODT, LEDTRT, LEDDSTRMG, LEDDOSE, LEDDOSFRQ, STARTDT, STOPDT, LEDD, ORIG_ENTRY, note_clean]
Index: []


NameError: name 'jaentl' is not defined

## Edit user error input

In [ ]:
# Dictionary of substring replacements
replacements = {
    "SINEMET 250100": "SINEMET 100/250",
    "SINEMET CR 250199": "SINEMET CR 199/250",
    "RYTARY 61.75/2445 MG": "RYTARY 61.75/245 MG",
    "CARBIDOPA LEVODOPA (25-1000": "CARBIDOPA LEVODOPA (25-100)",
    "CARBIDOPA LEVODOPA (25/1000": "CARBIDOPA LEVODOPA (25-100)",
    "CARBIDOPA/LEVODOPA ER (25/1000" : "CARBIDOPA LEVODOPA (25-100)",
    "CARBIDOPA LEVODOPA ER(25/1000": "CARBIDOPA LEVODOPA (25-100)",
    "LEVODOPA/BENSERAZID 100725": "LEVODOPA/BENSERAZID 100",
    "LEVODOPA BENSERAZID 100725": "LEVODOPA/BENSERAZID 100", 
    "CARBIDOPA LEVODOPA 25-10" : "CARBIDOPA LEVODOPA 25-100",
    "CARBIDOPA-LEVODOPA 25-10" :"CARBIDOPA LEVODOPA 25-100",
    "STALEVO 100\\25\\2000" : "STALEVO 100/25/200",
    "RYTARY (36-25-145)" : "RYTARY (36.25-145)", 
    "LEVODOPA CARBIDOPA  (20+5) INTESTINAL GEL" : "Duodopa (20+5)", 
    "LEVODOPA/CARBIDOPA INTESTINAL GEL INFUSION" : "Duodopa", 
    "PRAMIPEXOLE O,5 MG" : "PRAMIPEXOLE O.5 MG",
}

# Apply all replacements in one pass
df['LEDTRT'] = df['LEDTRT'].replace(replacements, regex=False)

df2 = df.copy()

## Normalize raw medication text
This function:

Standardizes formatting

Normalizes dose separators

Fixes decimal formatting

Removes noise

Prepares data for structured extraction

It’s a preprocessing step before NLP-style parsing.

In [ ]:
def normalize_text(x):
    if pd.isna(x):
        return None
    x = x.lower()
    x = x.replace('corbilta', '')
    x = x.replace('\\', '/')
    x = re.sub(r'\b7(?=[A-Za-z])', '', x) # FIX levodopa 7benserazid 100/25 
    x = x.replace('(2)', 'X 2') # for PRAMIPEXOLE (MIRAPEX) .125 MG (2) 
    x = x.replace(', ', ',')
    x = re.sub(r'(?<=\d),(?=\d)', '.', x)
    x = x.replace(',', ' ')  # replace commas with spaces
    x = x.replace('+', '/')
    x = re.sub(r'[^\w\s/.-]', '', x)  # remove other unwanted punctuation
    x = re.sub(r'\s+', ' ', x).strip()  # collapse multiple spaces
    x = re.sub(r'\s*/\s*', '/', x) # remove spaces before/after / 
    x = x.replace('//', '/')
    x = x.replace('-', '/')
    unicodedata.normalize("NFKC", str(x))
    return x



df["med_clean"] = df["LEDTRT"].apply(normalize_text)
df.to_csv('/Users/emudr/Desktop/x.csv')

## Extract structured components from medication text
 - frequency (e.g., 3/day)
 - drug formulation type (ER, CR, etc.)
 - dosage values


In [ ]:
def extract_med_components(text,  leddmstrmg=None, leddosfrq=None, leddose = None, ledtrt = None):
    """
    Extracts frequency, drug type, and dosage from pre-cleaned med text.
    Returns cleaned_text, frequency, drug_type, dosage_list
    If leddmstrmg is provided, it is favored over extracted dosage.
    """
    if pd.isna(text):
        return None, None, None, []

    clean_text = text


    #### Frequency (LEDDOSFRQ) --> Need to get rid of hours first 
    freq_match = re.search(r'(\d+\s*hours)', clean_text, re.IGNORECASE)
    freq = freq_match.group(1).strip() if freq_match else None

    # Preserve existing LEDDOSE if present
    if pd.notna(leddosfrq):
        freq = leddosfrq

    # Map extracted frequency to integer
    freq_map = {
        '24hours': 1,
        '24 hours' : 1
        }
    
    if freq is not None:
        freq = freq_map.get(freq, freq)  # Returns None if not in map
        
    clean_text = re.sub(r'(\d+\s*hours)', '', clean_text, flags=re.IGNORECASE) # remove extracted frequency 


    
    #### Drug release formula 
    er_terms = ["CR","ER","XR","XL","SR","DR","LA","SA","TR","lp", "modified release", "reseased lib", "depot", "lt", "odt",
                  "extended release","prolonged/release","sustained/release",
                  "sustained release","retard","slow release", 'ret.']
    ir_terms = ["IR"]
    
    # Create mapping dictionary
    drug_release_map = {term.lower(): "ER" for term in er_terms}
    drug_release_map.update({term.lower(): "IR" for term in ir_terms})
    

    # Build regex pattern from all mapping keys
    type_pattern = r"\b(" + "|".join(map(re.escape, drug_release_map.keys())) + r")\b"
    
    # Search in clean_text
    type_match = re.search(type_pattern, clean_text, flags=re.IGNORECASE)
    
    drug_type = None
    if type_match:
        matched_term = type_match.group(0).lower()
        drug_type = drug_release_map.get(matched_term, None)
        # Remove the original matched term from clean_text
        clean_text = re.sub(r"\b" + re.escape(type_match.group(0)) + r"\b", "", clean_text, flags=re.IGNORECASE)

    
        
    
   #### LEDDOSE (# of pills, tabs, or numbers before /day)
    dose = []
    
    dose_match = re.search(r'(\d+)\s*(?:pill|tabs?)', clean_text, re.IGNORECASE)
    if dose_match:
        dose.append(dose_match.group(1)) 
        clean_text = re.sub(r'(\d+)\s*(?:pill|tabs?)', '', clean_text, flags=re.IGNORECASE)
     
    dose_match = re.search(r'(\d+)(?=/day)', clean_text, re.IGNORECASE)
    if dose_match:
        dose.append(dose_match.group(1))
        clean_text = re.sub(r'(\d+)(?=/day)', '', clean_text, flags=re.IGNORECASE)
    
    # 4️⃣ Numbers after "X" or "x" (e.g., "X 8")
    x_matches = re.findall(r'[Xx]\s*(\d+)', clean_text)
    if x_matches:
        dose.extend(x_matches)
        clean_text = re.sub(r'[Xx]\s*\d+', '', clean_text)

    # Preserve existing LEDDOSE if present
    if pd.notna(leddose):
        dose.insert(0, str(leddose)) 

    dose = ','.join(dose) if dose else np.nan # remove [] 

    

    
    ##### LEDDSTRMG
    mg_pattern = r'\d*\.?\d+(?:/\d*\.?\d+)+|\d*\.?\d+\s*mg|\d*\.\d+|\d+'

    mg_matches = re.findall(mg_pattern, clean_text)
    mg_matches = [m.replace('mg','').strip() for m in mg_matches]
    mg_dose = '/'.join(map(str, set(mg_matches))) if mg_matches else np.nan
    
    # Remove matched values from clean_text
    clean_text = re.sub(mg_pattern, '', clean_text)
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()

    # Replace commas in LEDDSTRMG with '/' if LEDTRT contains 'and'
    if pd.notna(ledtrt) and 'and' in ledtrt.lower() and pd.notna(mg_dose) and ',' in mg_dose:
        mg_dose = mg_dose.replace(', ', '/')

        
    if pd.notna(mg_dose):
        # split and clean
        parts = [p.strip() for p in mg_dose.split('/')]
        
        # sort numerically
        parts = sorted(parts, key=lambda x: float(x))
        
        if len(parts) >= 3:
            # If the largest value is 200, remove it
            if parts[-1] == '200':
                parts = [parts[1]]  # keep only the middle value
            # Keep the two largest remaining values
            mg_dose = '/'.join(parts[-2:])
        elif len(parts) == 2:
            # Keep only the largest value
            mg_dose = parts[-1]
        else:
            # Single value
            mg_dose = parts[0]
        


        
    # LEDDOSFRQ
    if freq is None :
        freq_match = re.search(r'\b(day|tid)\b', clean_text, re.IGNORECASE)
        freq = freq_match.group(1).strip() if freq_match else None
    
        # Preserve existing LEDDOSE if present
        if pd.notna(leddosfrq):
            freq = leddosfrq
            
        if freq_match: 
            clean_text = re.sub(r'\b(day|tid)\b', '', clean_text, flags=re.IGNORECASE)
        
               # Map extracted frequency to integer
        freq_map = {
            'day': 1,
            'tid': 3
        }
    
        if freq is not None:
            freq = freq_map.get(freq, freq)  # Returns None if not in map
        
    
    
    return clean_text, freq, drug_type, mg_dose, dose 


df[['med_clean', 'LEDDOSFRQ', 'drug_release', 'LEDD_MG', 'LEDDOSE']] = df.apply(lambda row: extract_med_components(
        row['med_clean'],
        leddmstrmg=row.get('LEDDSTRMG'),
        leddosfrq=row.get('LEDDOSFRQ'), 
        leddose = row.get('LEDDOSE'), 
        ledtrt=row.get('LEDTRT'), 
    ),
    axis=1,
    result_type='expand'
)

df['LEDDSTRMG'] = df['LEDDSTRMG'].fillna(df['LEDD_MG'])

# Further clean medication names for matching 
 - remove formulation words
 - remove delivery method terms
This function is a drug name normalizer.

It takes messy medication strings and reduces them down to a clean, simplified drug name, removing dosage info, formulation terms, and brand noise.

In [ ]:
def preprocess_med_name(med):
    """
    Lowercase, remove extra words and punctuation, collapse spaces.
    """

    extra_words = ["mg", "patch", "transdermal", "system", "tablet", "capsules", "dispersible", "dispersable", "plaster", "hcl", "inhalation", "elixir", "hydrochloride", "generic", "pen", "pump", " and", "total daily dose", "liquid", "neuraxpharm", "leganto"]
    
    if not med or pd.isna(med):
        return ""
    
    med = med.lower()
    
    # Check for 'duodopa' first
    if 'duodopa' in med:
        return 'duodopa'
    
    # Remove punctuation
    med = re.sub(r'[-/(),]', ' ', med)
    
    # Remove extra words
    for word in extra_words:
        med = med.replace(word, "")
    
    # Collapse multiple spaces
    med = " ".join(med.split())
    
    return med

    
# Apply preprocessing to the column
df['med_clean'] = df['med_clean'].apply(preprocess_med_name)
df.to_csv("/Users/emudr/Desktop/temp.csv")



# Define canonical medication vocabulary and fuzzy-match cleaned meds to canonical names

In [ ]:
canonical_meds = [
    "amantadine",
    "apokyn",
    "apomorphine",
    "azilect",
    "azilect rasagiline",
    "carbidopa levodopa entacapone",
    "carbidopa levodopa",
    "carbidopa levodopa sinemet",
    "clarium",
    "co carledopa",
    "comtan",
    "dopicar", 
    "duodopa",
    "eldepryl",
    "entacapone",
    "osmolex amantadine",
    "gocovri amantadine", 
    "gocovri",
    "inbrija",
    "isicom",
    "istradefylline", 
    "kynmobi",
    "levocomp",
    "levodopa",
    "levodopa benserazide",
    "madopar",
    "mirapex",
    "nacom",
    "neupro",
    "opicapona",
    "osmolex",
    "piribedil",
    "pk merz",
    "pramipexole",
    "pramipexole mirapex",
    "rasagiline",
    "requip",
    "ropinirole",
    "rotigotine",
    "ongentys",
    "rytary",
    "rytary carbidopa levodopa", 
    "safinamide",
    "selegiline",
    "sifrol",
    "sinemet",
    "sinemet carbidopa levodopa",
    "sinemet plus",
    "stalevo",
    "tolcapone",
    "trihexyphenidyl",
    "xadago"
]

from thefuzz import fuzz, process


def fuzzy_match_best(med, canonical_list, threshold=80):
    """
    Match med name to canonical list, avoiding over-matching to longer combinations.
    """
    if not med or pd.isna(med):
        return ("", 0)
    
    med_clean = " ".join(str(med).lower().split())
    
    # exact match first
    for canon in canonical_list:
        if med_clean == canon.lower():
            return (canon, 100)
    
    candidates = process.extract(
        med_clean,
        canonical_list,
        scorer=fuzz.token_set_ratio,
        limit=None
    )
    
    # filter by threshold
    candidates = [(c, s) for c, s in candidates if s >= threshold]
    if not candidates:
        return (med, 0)
    
    # penalize candidates with extra tokens not in input
    def score_penalty(candidate):
        c_name, score = candidate
        c_tokens = set(c_name.lower().split())
        med_tokens = set(med_clean.split())
        extra_tokens = len(c_tokens - med_tokens)
        # subtract 5 points per extra token
        return (score - 5 * extra_tokens, -extra_tokens)
    
    candidates.sort(key=score_penalty, reverse=True)
    
    best_match, best_score = candidates[0]
    return (best_match, best_score)

df['med_match'] = df['med_clean'].apply(lambda x: fuzzy_match_best(x, canonical_meds))
df[['med_canonical', 'match_score']] = pd.DataFrame(df['med_match'].tolist(), index=df.index)
temp = df[['LEDTRT', 'med_clean', 'med_canonical', 'match_score', 'drug_release', 'LEDDSTRMG']]
#df.drop(columns=['med_match'], inplace=True)
temp.to_csv(outdir + 'fizz.csv')

# Manually resolve remaining unmatched or ambiguous meds


In [ ]:
manual_map = { 
    "carb levo": "carbidopa levodopa",
    "ropirinol": "ropinirole",
    "tasmar": "tolcapone",
    "l dopa": "levodopa",
    "dopa": "levodopa",
    "apomorfina": "apomorphine",
    "liquid amantadine" : "amantadine", 
    "benserazida": "levodopa benserazide",
    "levopar plus": "levodopa benserazide"
}

df["med_canonical"] = df["med_canonical"].map(manual_map).fillna(df['med_canonical'])


# Map canonical medications to standardized drug names
 - Translate med_canonical to standardized drug labels

In [ ]:
med_canonical_2_drug = {
    # Amantadine
    "amantadine": "amantadine",
    "gocovri amantadine" : "gocovri", 
    "osmolex amantadine" : "osmolex",
    "gocovri": "gocovri",
    "osmolex": "osmolex",
    "pk merz": "amantadine",

    # Apomorphine
    "apomorphine": "apomporphine",
    "apokyn": "apomporphine",

    # Carbidopa/Levodopa + Entacapone
    "carbidopa levodopa entacapone": "carbidopa levodopa entacapone",
    "stalevo": "carbidopa levodopa entacapone",

    # Carbidopa / Levodopa
    "sinemet": "carbidopa/levodopa",
    "carbidopa levodopa": "carbidopa/levodopa",
    "sinemet carbidopa levodopa": "carbidopa/levodopa",
    "carbidopa levodopa sinemet": "carbidopa/levodopa",
    "nacom": "carbidopa/levodopa",
    "levocomp": "carbidopa/levodopa",
    "sinemet plus": "carbidopa/levodopa",
    "dopicar": "carbidopa/levodopa",
    "isicom": "carbidopa/levodopa",

    # Controlled-release levodopa
    "rytary": "controlled-release levodopa",
    "rytary carbidopa levodopa": "controlled-release levodopa",
    "co carledopa": "controlled-release levodopa",

    # COMT inhibitors
    "entacapone": "entacapone",
    "comtan": "entacapone",

    # Inhaled levodopa
    "inbrija": "inbrija",

    # Adenosine A2A antagonist
    "istradefylline": "istradefylline",

    # Levodopa / Benserazide
    "levodopa": "levodopa",
    "madopar": "levodopa",
    "levodopa benserazide": "levodopa",
    "duodopa" : "duodopa", 

    # Opicapone
    "opicapona": "opicapone",
    "ongentys": "opicapone",

    # Other
    "trihexyphenidyl": "other",

    # Piribedil
    "clarium": "piribedil",
    "piribedil": "piribedil",

    # Pramipexole
    "pramipexole": "pramipexole",
    "pramipexole mirapex": "pramipexole",
    "mirapex": "pramipexole",
    "sifrol": "pramipexole",

    # Rasagiline
    "azilect": "rasagiline",
    "azilect rasagiline": "rasagiline",
    "rasagiline": "rasagiline",

    # Ropinirole
    "ropinirole": "ropinirole",
    "requip": "ropinirole",

    # Rotigotine
    "rotigotine": "rotigotine",
    "neupro": "rotigotine",

    # Safinamide
    "xadago": "safinamide",
    "safinamide": "safinamide",

    # Selegiline
    "selegiline": "selegiline",
    "eldepryl": "selegiline",

    # Tolcapone
    "tolcapone": "tolcapone",
}

df["med_canonical"] = df["med_canonical"].map(med_canonical_2_drug)

# Map drugs to LEDD categories (drug classes)
 - Group drugs by mechanism / LEDD type

In [ ]:
# Map meds to LEDD type / category
ledd_category_map = {
    
    # Dopamine Replacement 
    'carbidopa levodopa entacapone': 'comt_inhibitor',
    'carbidopa/levodopa' : "dopamine_replacement", 
    'controlled-release levodopa' : "dopamine_replacement", 
    'levodopa' : "dopamine_replacement", 
    'inbrija' : "dopamine_replacement", 
    'duodopa' : 'dopamine_replacement', 

    # COMT inhibitors alone (0 LEDD, modify levodopa)
    "entacapone": "comt_inhibitor",
    "opicapone": "comt_inhibitor",
    "tolcapone": "comt_inhibitor",

    # Dopamine agonists (non-levodopa multipliers)
    "pramipexole": "dopamine_agonist",
    "ropinirole": "dopamine_agonist",
    "rotigotine": "dopamine_agonist",
    "piribedil": "dopamine_agonist",

    # MAO-B inhibitors (fixed additive LEDD)
    "selegiline": "mao_b_inhibitor",
    "rasagiline": "mao_b_inhibitor",
    "safinamide": "mao_b_inhibitor",
    "istradefylline": "moab_inhibitor",
    
    # Other drugs → 0 LEDD
    "amantadine": "other",
    "apomporphine": "other",
    "gocovri" : "other", 
    "osmolex" : "other"
}

df['drug_class'] = df['med_canonical'].map(ledd_category_map)

# Visualize distribution of LEDD categories

In [ ]:
# Count how many rows are in each LEDD category
counts = df['drug_class'].value_counts()

# Pie chart
plt.figure(figsize=(8,8))
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=140, colors=plt.cm.tab20.colors)
plt.title('Distribution of LEDD Categories')
plt.axis('equal')  # Equal aspect ratio ensures pie is circular
plt.show()

# Map drugs to conversion factors for LEDD calculation

In [ ]:
conversion_factor_map = {

    # Dopamine Replacement 
    'carbidopa levodopa entacapone': 0.33,
    'carbidopa/levodopa' : 1, 
    'controlled-release levodopa' : 0.7,
    'levodopa' : 1,
    'inbrija' : 0.6,
    'duodopa' : 1.11, 

    # COMT inhibitors alone (0 LEDD, modify levodopa)
    "entacapone": 0.33, # FIXME
    "opicapone": 0.33, # FIXME
    "tolcapone": 0.5, # FIXME

    # Dopamine agonists (non-levodopa multipliers)
    "pramipexole": 100, 
    "ropinirole": 20,
    "rotigotine": 30.3, 
    "piribedil": 1,

    # MAO-B inhibitors (fixed additive LEDD)
    "selegiline": 10, 
    "rasagiline": 100, 
    "safinamide": 1, 
    "istradefylline": 2.5, 
    
    # Other drugs
    "amantadine": 1, 
    "apomporphine": 10, 
    "gocovri" : 1.46,
    "osmolex" : 1
}

df['conversion_factor'] = df['med_canonical'].map(conversion_factor_map)

# Set conversion_factor = 0.7 for carbidopa/levodopa ER
df.loc[(df['med_canonical']== 'carbidopa/levodopa') & (df['drug_release'] == 'ER'),'conversion_factor'] = 0.7

## Clean LEDD column 

In [ ]:
df['LEDD'] = df['LEDD'].replace('LD x 0.33', np.nan)


def eval_ledd_expr(val):
    if pd.isna(val):
        return np.nan
    
    if not isinstance(val, str):
        return val
    
    # remove '+ LD' (with or without spaces)
    clean = re.sub(r'\+\s*LD', '', val, flags=re.IGNORECASE)
    
    # remove parentheses
    clean = clean.replace('(', '').replace(')', '')
    
    # normalize multiplication symbol
    clean = clean.replace('x', '*').replace('X', '*')
    
    try:
        return eval(clean)
    except Exception:
        return np.nan

df['LEDD'] = df['LEDD'].apply(eval_ledd_expr)

## Calcualte LEDD

In [ ]:
def to_float_safe(x):
    try:
        return float(x)
    except (ValueError, TypeError):
        return np.nan

# Apply to the column
df['LEDDSTRMG'] = df['LEDDSTRMG'].apply(to_float_safe)
df['LEDDOSE'] = df['LEDDOSE'].apply(to_float_safe)


df['LEDD_calculated'] = df['LEDDSTRMG'] * df['LEDDOSE'].fillna(1.0) * df['LEDDOSFRQ'].fillna(1.0) * df['conversion_factor']
#df['LEDD'] = df['LEDD'].fillna(df['LEDD_calculated'])
x = df[['LEDTRT', 'LEDDSTRMG', 'LEDDOSE', 'LEDDOSFRQ', 'med_canonical', 'conversion_factor', 'LEDD_calculated', 'LEDD']]
x.to_csv('/Users/emudr/Desktop/final.csv')


## Save Final dataset 

In [ ]:
# Fix up col name order 
df = df[['REC_ID', 'PATNO', 'EVENT_ID', 'INFODT', 'LEDTRT', 'med_clean', 'med_canonical',  'LEDDSTRMG', 'LEDDOSE', 'LEDDOSFRQ',
'drug_class', 'conversion_factor', 'LEDD_calculated', 'LEDD', 'STARTDT', 'STOPDT', 'ORIG_ENTRY']]

print(df['LEDD'].unique())
df.to_csv('/Users/emudr/Desktop/data/PPMI_LEDD_Project/simulate_data.csv')
janetl

In [ ]:
# Replace 'OTHER_COL' with the name of the column you want to include
df_unique = df[['LEDTRT', 'LEDDSTRMG', 'LEDDOSE', 'LEDDOSFRQ' ,'med_canonical', 'conversion_factor']].dropna(subset=['LEDDSTRMG'])

# Save to CSV
df_unique.to_csv('/Users/emudr/Desktop/help.csv', index=False)


In [ ]:
df['LEDD'].notna()

In [ ]:
df['LEDD'] = df['LEDD'].apply(to_float_safe)
print(sorted(df['LEDD'].unique()))

In [ ]:
new_df = df[df['LEDD'].notna()]
new_df.to_csv('/Users/emudr/Desktop/new_df.csv')
